In [2]:
import mediapy
import numpy as np
from tqdm import tqdm

from robokit.datasets.tcl_datasets import TCLDataset
from robokit.debug_utils import print_batch
from robokit.debug_utils import concatenate_rgb_images, plot_action_wrt_time, plot_force_sensor_wrt_time


In [8]:
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1009_spoon_pick_place/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1010_sweep_bean/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1010_pour_bean/")
dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1010_wipe_white_board/")
print("Total tasks:", len(dataset.tasks))
print_batch('TCLDataset', dataset[0])

[TCLDataset] total length: 1665
Total tasks: 8
TCLDataset: Dict, keys=['primary_rgb', 'gripper_rgb', 'primary_depth', 'gripper_depth', 'language_text', 'actions', 'rel_actions', 'robot_obs', 'force_torque']
--primary_rgb, <class 'numpy.ndarray'>, shape=(480, 848, 3), min=0.0000, max=235.0000, dtype=uint8
--gripper_rgb, <class 'numpy.ndarray'>, shape=(480, 848, 3), min=0.0000, max=255.0000, dtype=uint8
--primary_depth, <class 'numpy.ndarray'>, shape=(480, 848), min=0.0000, max=15363.0000, dtype=uint16
--gripper_depth, <class 'numpy.ndarray'>, shape=(480, 848), min=0.0000, max=54625.0000, dtype=uint16
--language_text: <class 'str'>, len=37, value='Wiping the writing on the whiteboard.'
--actions, <class 'numpy.ndarray'>, shape=(7,), min=0.0000, max=0.0000, dtype=float64
--rel_actions, <class 'numpy.ndarray'>, shape=(7,), min=0.0000, max=0.0000, dtype=float64
--robot_obs, <class 'numpy.ndarray'>, shape=(14,), min=-2.9306, max=3.0898, dtype=float64
--force_torque, <class 'numpy.ndarray'>, 

In [9]:
global_idx = 0

watch_idx = task_skip_idx = 7
vis_max_frames = 3500
fps = 10
sample_rate = 1  # 采样率，每sample_rate帧取一帧，以减少内存使用。例如，3表示每3帧取1帧。

# task_skip_idx = 35
# fps = 5

print("Total tasks:", len(dataset.tasks))

task_instruction = None
for task_idx, task in enumerate(dataset.tasks):
    task_length = min(vis_max_frames, dataset.task_lengths[task_idx])
    effective_length = (task_length + sample_rate - 1) // sample_rate  # 计算有效帧数，确保覆盖整个长度

    # 只存储必要的列表，移除未使用的images_primary和images_gripper
    images_cat = []
    actions = []
    force_torque = []

    if task_idx < task_skip_idx:
        global_idx += dataset.task_lengths[task_idx]
        continue

    for i in tqdm(range(effective_length)):
        # 计算实际索引，实现间隔抽帧
        actual_frame_idx = i * sample_rate
        if actual_frame_idx >= task_length:
            break  # 防止越界

        frame_data = dataset[global_idx + actual_frame_idx]

        images_cat.append(concatenate_rgb_images(frame_data['primary_rgb'], frame_data['gripper_rgb'], vertical=True))
        actions.append(frame_data['rel_actions'])
        force_torque.append(frame_data['force_torque'])

        task_instruction = frame_data['language_text']

    # 在循环后更新global_idx到下一个task
    global_idx += dataset.task_lengths[task_idx]

    all_vis = []
    actions_vis, fig, ax = plot_action_wrt_time(np.array(actions))
    force_torque_vis, _, _ = plot_force_sensor_wrt_time(np.array(force_torque))

    for frame_idx in range(len(images_cat)):  # 使用effective_length的长度
        # 先拼接相机图像和动作可视化
        temp_vis = concatenate_rgb_images(
            images_cat[frame_idx],
            actions_vis[frame_idx],
            vertical=False,
            resize_ratio=1
        )
        # 再拼接力传感器可视化
        all_vis.append(concatenate_rgb_images(
            temp_vis,
            force_torque_vis[frame_idx],
            vertical=False,
            resize_ratio=1
        ))

    print("Instruction:", task_instruction)
    mediapy.show_video(all_vis, fps=fps / sample_rate)  # 可选：调整fps以匹配采样率，保持视频时长类似

    break

Total tasks: 8


100%|█████████████████████████████████████████| 501/501 [00:08<00:00, 59.24it/s]


Plotting action dynamic figures...
Plotting force sensor dynamic figures...
Instruction: Wiping the writing on the whiteboard.


This browser does not support the video tag.


In [4]:
dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1009_spoon_pick_place")
global_idx = 0

# task_skip_idx = 2
# fps = 30

task_skip_idx = 3
fps = 30

print("Total tasks:", len(dataset.tasks))

for task_idx, task in enumerate(dataset.tasks):
    task_length = dataset.task_lengths[task_idx]
    images_primary, images_gripper = [], []
    images_cat = []
    actions = []

    if task_idx < task_skip_idx:
        global_idx += task_length
        continue

    for frame_idx in tqdm(range(task_length)):
        frame_data = dataset[global_idx]
        global_idx += 1

        images_primary.append(frame_data['primary_rgb'])
        images_gripper.append(frame_data['gripper_rgb'])
        images_cat.append(concatenate_rgb_images(frame_data['primary_rgb'], frame_data['gripper_rgb'], vertical=True))
        actions.append(frame_data['rel_actions'])

    all_vis = []
    actions_vis, fig, ax = plot_action_wrt_time(np.array(actions))
    for frame_idx in range(task_length):
        all_vis.append(concatenate_rgb_images(images_cat[frame_idx], actions_vis[frame_idx],
                                              vertical=False, resize_ratio=1))

    mediapy.show_video(all_vis, fps=fps)

    break

[TCLDataset] total length: 28589
Total tasks: 50


100%|███████████████████████████████████████████████████████████████████████████████████| 455/455 [00:08<00:00, 53.70it/s]


Plotting action dynamic figures...


This browser does not support the video tag.
